# 41장 · 같은 질문으로 LCEL과 그래프의 실험 비교하기

“B형은 무선으로 파일을 보낼 수 없다”와 “제공된 설명서에서는 무선 전송을 확인할 수 없다”는 사용자에게 서로 다른 안내입니다. 앞 문장은 기능이 없다고 단정합니다. 뒤 문장은 알고 있는 범위를 알려 줍니다. 두 답 모두 케이블 복사를 안내하는 2쪽을 찾을 수 있습니다. **검색 성공과 답변 성공을 어떻게 나누어 살펴볼까요?**

이번에는 24장의 LCEL 체인, 39장의 그래프, 검색 후보 수 하나를 바꾼 그래프를 같은 개발용 질문으로 실행합니다. 어떤 자료와 설정을 고정했는지, 출력이 달라진 첫 단계는 어디인지, 한 차례 실험으로 결론을 내릴 수 있는지 판단합니다.

**준비물**. 전체 실습 ZIP을 풀고 Jupyter 커널을 그 환경으로 선택합니다. 앞 장에서 쓰던 패키지에 LangSmith SDK가 필요합니다. 설치가 필요한 커널에서는 별도 셀에 `%pip install "langsmith>=0.4,<1" langchain-openai langgraph chromadb pypdf python-dotenv streamlit`를 실행한 뒤 커널을 재시작하세요. 압축을 푼 폴더의 `.env`에 개인 실습용 `LANGSMITH_API_KEY`, `OPENAI_API_KEY`, 세 장에서 사용할 같은 `OPENAI_MODEL`을 둡니다. LangSmith 워크스페이스가 여럿이면 해당 워크스페이스의 키 또는 `LANGSMITH_WORKSPACE_ID`도 확인하세요. 팀이 선택한 LangSmith 엔드포인트가 있다면 `LANGSMITH_ENDPOINT`도 맞춥니다. 모델 호출은 계정 사용량에 반영됩니다. 원격 기록에는 제공된 가상 제품 자료를 사용합니다.

In [ ]:
from pathlib import Path
import json
import os
import sys
from dotenv import load_dotenv

# 전체 실습 묶음을 푼 폴더 또는 그 안의 장 폴더에서 노트북을 엽니다.
base = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pre-mid/documents/evaluation/eval_lab.py").is_file())
docs = base / "pre-mid/documents"
sys.path.insert(0, str(docs / "evaluation"))
load_dotenv(base / ".env")
from eval_lab import examples

# 키의 실제 값은 출력하지 않습니다. 개인 실습 공간의 키를 사용하세요.
smith_ready = bool(os.getenv("LANGSMITH_API_KEY") or os.getenv("LANGCHAIN_API_KEY"))
model_ready = bool(os.getenv("OPENAI_API_KEY"))
if smith_ready:
    os.environ["LANGSMITH_TRACING"] = "true"
    from langsmith import Client
    client = Client()
work = base / "my-recorder-evaluation"
work.mkdir(exist_ok=True)
print("LangSmith 연결 설정", smith_ready, "/ 모델 연결 설정", model_ready)

## 1. 비교하기 전에 기준선과 가설을 적기

**기준선**은 변경 효과를 판단할 때 되돌아볼, 설정을 기록한 현재 버전입니다. **실험**은 평가 데이터셋의 각 입력을 같은 한 버전에 보내 결과와 점수를 묶은 기록입니다. 오늘은 실험 세 개를 만듭니다.

| 이름 | 학생이 앞에서 만든 구현 | 확인할 질문 |
|---|---|---|
| lcel-k4 | 24장의 검색·답변 체인, 후보 네 쪽 | 검색된 각 쪽을 그대로 답변에 쓸 때 어떤 실수가 생기는가? |
| graph-k4 | 39장의 근거 판정·재검색 그래프, 후보 네 쪽 | 같은 자료에서 기종과 근거 판정이 보류에 도움이 되는가? |
| graph-k2 | 같은 그래프, 후보 두 쪽 | 후보를 줄여도 올바른 쪽이 남는가? 재검색 때문에 시간이 오히려 늘어나는가? |

첫 비교는 구조 자체가 달라 어느 노드 하나의 효과라고 단정하기 어렵습니다. 두 번째 비교에서는 후보 수만 바꾸며 개발 질문, 실제 PDF, 색인, 모델, 프롬프트 코드를 고정합니다. 수업의 기종과 판본은 질문 문장에 명시했고, 그래프는 같은 내용을 화면의 제품 선택값으로도 받습니다. 정답 쪽수는 어느 구현에도 전달하지 않습니다.

In [2]:
dev = examples("dev")
question = dev[-1]["inputs"]["question"]
print(question)
print("기대하는 행동", dev[-1]["outputs"]["action"])

2027년 5월 기준으로 B형에서 녹음 파일을 무선으로 보내는 방법이 있나요?
기대하는 행동 abstain


D05처럼 답을 보류해야 하는 질문에는 정답 페이지가 없습니다. ‘틀린 페이지’라는 이유로 모든 문서 순위를 0으로 처리하면 보류 질문이 많을수록 검색기가 나쁜 것처럼 보입니다. 순위는 정답 페이지가 있는 질문에서만 계산하고, 보류 여부는 별도 열에 남깁니다.

## 2. 점수 세 개가 서로 대답하는 질문

정답 쪽이 최종 후보에서 세 번째라면 역순위(reciprocal rank)는 1/3입니다. 찾지 못하면 0입니다. 기종·판본·쪽은 모두 같아야 합니다. 재검색한 그래프에서 이 지표는 **마지막 검색**을 봅니다. 첫 검색에서 빠졌다면 같은 실행의 searches에서 두 후보 목록을 직접 비교하세요.

답에 붙인 인용 목록 안에 정답 쪽이 있는지, ‘답하기/보류’가 참고 답의 행동과 일치하는지도 별도로 측정합니다. 둘 다 통과해도 수치가 틀릴 수 있습니다. 45분이라고 써야 하는 답을 30분으로 잘못 써 놓고 1.1판 2쪽만 정확히 붙일 수 있기 때문입니다.

In [3]:
from eval_lab import assessment
reference = dev[3]["outputs"]
# 사람이 만든 채점기 점검용 반례입니다. 인용은 맞고 숫자는 틀립니다.
counterexample = {"action": "answer", "answer": "30분마다 나뉩니다.",
    "candidates": ["recorder-b-1.0.pdf#p2", "recorder-a-1.1.pdf#p2"],
    "citations": ["recorder-a-1.1.pdf#p2"]}
print(assessment(counterexample, reference)["results"])

[{'key': 'action_match', 'score': 1}, {'key': 'retrieval_rr', 'score': 0.5}, {'key': 'citation_has_gold', 'score': 1}]


실제로 계산하면 행동은 1, 역순위는 0.5, 인용 포함 여부는 1입니다. 그런데 4월 A형은 45분마다 나뉩니다. 사람 검토에는 ‘시간이 정확한가’와 ‘파일이 바뀌어도 녹음이 계속된다는 조건이 있는가’를 써야 합니다. 두 인용 중 하나만 맞는 경우까지 인용 포함 점수가 1일 수 있으니 여분의 인용도 읽습니다.

## 3. 집필 때 실제로 관찰한 응답으로 읽는 법 익히기

실습 파일 `dev-observed.json`은 제공된 3개 PDF와 개발 질문 5개를 OpenAI 모델에 실제로 보내 얻은 15건의 로컬 실행 기록입니다. 개인 LangSmith 계정에서 생성한 실험은 아래에서 별도로 실행합니다. 같은 모델 이름과 설정으로 다시 실행해도 표현과 시간은 달라질 수 있습니다. 각 행의 settings에는 당시 색인과 코드·프롬프트 식별값이 있습니다. 현재 실습 출력과 버전이 다르면 그 차이도 적으세요.

In [4]:
observed = json.loads((docs / "evaluation/dev-observed.json").read_text(encoding="utf-8"))
for row in observed["rows"]:
    if row["id"] == "D05":
        print(row["version"], "→", row["output"]["answer"])

lcel-k4 → 2027년 5월 기준으로 B형에서 녹음 파일을 무선으로 보내는 방법은 없습니다. 자료에 따르면 B형 1.0판(2027년 4월 1일부터 적용)에서는 녹음 파일을 컴퓨터로 옮길 때 케이블로 연결하여 RECORD 폴더에서 복사하는 방법만 안내되어 있습니다. 무선 전송에 관한 내용은 전혀 언급되어 있지 않습니다.[recorder-b-1.0.pdf#p2]
graph-k4 → 선택한 설명서에서 답을 확인할 근거가 부족합니다.
graph-k2 → 선택한 설명서에서 답을 확인할 근거가 부족합니다.


이 기록에서 LCEL의 답은 “무선으로 보내는 방법은 없습니다”로 시작합니다. 원문에는 케이블 복사 방법이 있을 뿐 무선 기능이 없다는 근거가 없습니다. 두 그래프는 확인할 근거가 부족하다고 답했습니다. **이 질문에서** 보류 행동이 개선됐다는 증거입니다. 후보 수를 4에서 2로 줄인 덕분이라고 말할 수 있을까요? 두 그래프의 D05가 모두 보류했으므로 이 사례는 그 주장을 지지하지 않습니다.

## 실습 · 개발 실험 하나를 원문으로 감사하기

짝은 15건 중 D03 또는 D04의 임의 버전을 고르고 **버전 이름을 가린 채** 답만 읽어 줍니다. 듣는 사람은 맞는 간격과 녹음 지속 여부를 먼저 판정하세요. 다음에 검색된 쪽과 실제 인용을 열어 같은 판단이 유지되는지 봅니다. 3월판과 4월판의 차이가 중요하므로 날짜가 보이지 않는 페이지 번호만으로 결정할 수 없습니다.

마지막으로 D05의 LCEL 답을 읽습니다. 어느 한 문장이 근거의 범위를 넘었나요? 파일과 답의 해당 구절을 두 열에 적으세요. 정답 단어 포함률을 세는 채점기라면 이 문제를 놓칠 이유를 동료에게 설명합니다.

In [5]:
# D03의 관찰 응답을 골라 PDF와 비교하세요. 다른 버전을 볼 때는 version만 바꿉니다.
picked = next(row for row in observed["rows"] if row["id"] == "D03" and row["version"] == "graph-k4")
print(picked["output"]["answer"], picked["output"]["citations"])

30분마다 나뉩니다. ['recorder-a-1.0.pdf#p2']


<details class="notebook-answer"><summary>예시 풀이 열기</summary><pre><code>picked = next(row for row in observed[&quot;rows&quot;] if row[&quot;id&quot;] == &quot;D05&quot; and row[&quot;version&quot;] == &quot;lcel-k4&quot;)
print(picked[&quot;output&quot;][&quot;answer&quot;])</code></pre><p>D03은 2027년 3월 A형으로, 1.0판 2쪽에 30분이라고 적혀 있습니다. D05의 경우 “케이블로 복사”에 대한 근거가 있어도 “무선으로 보내는 방법은 없다”는 단정은 근거가 없습니다. 답이 원문의 단어를 많이 공유한다는 사실은 이 논리적 차이를 검사하지 못합니다.</p></details>

## 4. 내 워크스페이스에 같은 질문으로 세 실험 남기기

먼저 하나의 색인을 열고 평가 대상을 만듭니다. 평가 대상 함수는 inputs만 받아 실제 챗봇을 부릅니다. 그 뒤에 평가자가 입력과 분리된 참고 답을 받아 점수를 매깁니다. helper 안의 `lcel_result`는 생성된 답에 쓰인 대괄호 인용을 읽고, `graph_result`는 그래프가 실제로 반환한 citations와 경로를 읽습니다. 검색 후보를 그대로 답의 인용으로 복사하면 안 됩니다.

In [ ]:
from eval_lab import make_targets, compare
live = smith_ready and model_ready
if live:
    from rag_lab import open_index
    index = open_index()
    targets = make_targets(index, candidate_k=2)
    print("색인 버전", index[0].name, "/ 비교 대상", list(targets))

다음 셀은 개발용 질문 5개를 구현 세 개에 각각 보냅니다. 그래프는 재검색과 생성 때문에 질문 한 건에서 모델을 여러 번 부를 수 있습니다. 실행 결과에는 실험 이름과 ID, 각 질문의 run ID, 설정, 응답 및 평가자 결과가 담깁니다. 중간에 멈췄다면 완료된 실험이 무엇인지 워크스페이스에서 먼저 확인하고, 필요한 대상만 `targets`에서 골라 재실행합니다.

In [ ]:
if live:
    report = compare(client, "dev", dev, targets, index)
    (work / "dev-results.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    for name, item in report.items():
        print(name, item["experiment"], item["experiment_id"])
        print("오류가 기록된 질문", [r["id"] for r in item["rows"] if r["error"]])

Datasets & Experiments에서 개발 세트를 열고 두 그래프의 실험을 함께 선택합니다. D05 행의 Actual Output과 Reference Output을 비교한 뒤 해당 run을 엽니다. 재작성된 검색어가 있다면 기종·요구 동작·사용 시점 중 무엇을 보존했는지 찾습니다. 다른 답을 낸 행이 없다면 후보 순서나 재검색 횟수가 달라진 행을 엽니다. 똑같은 결과는 그대로 ‘차이 없음’으로 기록합니다.

실패한 호출도 행 수에 포함해 별도 기록합니다. 실행 오류로 비어 있는 답을 ‘정상 보류’로 세면 모델 접속 장애가 우수한 평가로 보일 수 있습니다.

In [ ]:
if live:
    for name, item in report.items():
        # LangSmith가 집계한 지연과 토큰입니다. 수집되지 않은 비용은 None으로 남깁니다.
        stats = client.read_project(project_id=item["experiment_id"], include_stats=True)
        print(name, "지연 중앙값(초)", stats.latency_p50.total_seconds() if stats.latency_p50 is not None else None, "총 토큰", stats.total_tokens)

한 번 측정한 지연에는 네트워크, 순서, 최초 색인 준비 등도 영향을 줄 수 있습니다. 모델 답이 같고 시간이 조금 다르다면 고정된 입력에서 여러 번 실행해 분포를 봅니다. 총 토큰이 비었으면 0원으로 해석할 수 없습니다. 추적된 모델 호출의 usage를 열어 해당 모델·SDK 조합에서 수집됐는지 확인합니다. 실행 건수가 다를 때 총합만 비교하면 질문 한 건당 비용도 잘못 읽게 됩니다.

## 개인 실습 · 한 가지 설정 변경에 증거 붙이기

먼저 40장의 진단을 펼쳐 후보 수를 줄이면 좋아질 것으로 예상하는 행, 나빠질 수 있는 행을 하나씩 고릅니다. 본인 실험의 후보·인용·보류·실제 문장을 원문과 대조하세요. D04는 답을 직접 읽어 ‘45분’과 ‘녹음 지속’ 두 조건을 채점하고, D05는 기능 부재 단정과 근거 부족 안내를 구별합니다. 근거가 되는 구절, 실험 이름, 해당 run ID를 활동지에 넣습니다.

독립 과제는 동료가 고른 질문으로 **동료의 결론을 반박할 수 있는 증거를 하나 찾는 것**입니다. 차이가 없다면 그 사실을 기록하세요. 마지막 확인에 넘길 k와 판정 기준은 지금 적어 두세요. 확인용 질문은 아직 열 필요가 없습니다.

In [ ]:
# 동일한 정답 쪽을 인용해도 틀릴 수 있는 다른 답을 직접 써 보세요.
my_test_answer = ""

<details class="notebook-answer"><summary>모범 예시와 판정 근거 보기</summary><pre><code>my_test_answer = &quot;45분마다 녹음이 멈춥니다.&quot;
print(my_test_answer)</code></pre><p>D04의 ‘45분마다 녹음이 멈춘다’는 분할 간격과 인용 쪽이 맞아도 틀립니다. 1.1판 2쪽에는 파일이 바뀌어도 녹음은 계속된다고 쓰여 있습니다. D05에 대한 관찰은 두 그래프가 같은 행동이므로 k=2 채택의 근거가 될 수 없습니다.</p></details>

**다음 장에 가져갈 것**. 팀이 선택한 그래프 후보, 개발 실험 두 개의 ID, 모델과 색인 버전, 사람이 확인할 근거 조건을 기록합니다. 42장에서는 같은 기준을 한 번도 사용하지 않은 질문에 적용합니다.

[LangSmith 실험 실행](https://docs.langchain.com/langsmith/evaluate-llm-application) · [대상 함수에 전달하는 입력](https://docs.langchain.com/langsmith/define-target-function)